In [1]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv("../DAY 1/.env")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Gemini connected!")

Gemini connected!


In [2]:
resume_text = """
John Anderson
Email: john.anderson@gmail.com
Phone: +91 9876543210

Education:
Master of Computer Applications (MCA), ABC University, 2024

Skills:
Python, SQL, Flask, FastAPI, Git, Docker

Experience:
Worked as a Python Developer Intern at TechNova for 6 months.
Built REST APIs using Flask and worked with MySQL databases.
"""

In [17]:
from pydantic import BaseModel

class Education(BaseModel):
    degree: str
    year: str

class Experience(BaseModel):
    role: str
    description: str

class Resume(BaseModel):
    name: str
    email: str
    skills: list[str]
    education: list[Education]
    experience: list[Experience]

In [18]:
prompt = f"""
Extract the information from the resume below.

Return these fields:
- name
- email
- skills
- education
- experience

Resume:
{resume_text}
"""

In [19]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)

print(response.text)

{
  "name": "John Anderson",
  "email": "john.anderson@gmail.com",
  "skills": [
    "Python",
    "SQL",
    "Flask",
    "FastAPI",
    "Git",
    "Docker"
  ],
  "education": [
    {
      "degree": "Master of Computer Applications (MCA)",
      "institution": "ABC University",
      "year": "2024"
    }
  ],
  "experience": [
    {
      "role": "Python Developer Intern",
      "company": "TechNova",
      "duration": "6 months",
      "description": "Built REST APIs using Flask and worked with MySQL databases."
    }
  ]
}


In [20]:
import json

data = json.loads(response.text)

resume = Resume.model_validate(data)

print(resume)

name='John Anderson' email='john.anderson@gmail.com' skills=['Python', 'SQL', 'Flask', 'FastAPI', 'Git', 'Docker'] education=[Education(degree='Master of Computer Applications (MCA)', year='2024')] experience=[Experience(role='Python Developer Intern', description='Built REST APIs using Flask and worked with MySQL databases.')]


In [21]:
print("Name:", resume.name)
print("Email:", resume.email)
print("Skills:", resume.skills)
print("Education:", resume.education)
print("Experience:", resume.experience)

Name: John Anderson
Email: john.anderson@gmail.com
Skills: ['Python', 'SQL', 'Flask', 'FastAPI', 'Git', 'Docker']
Education: [Education(degree='Master of Computer Applications (MCA)', year='2024')]
Experience: [Experience(role='Python Developer Intern', description='Built REST APIs using Flask and worked with MySQL databases.')]


In [22]:
bad_data = {
    "name": "John",
    "email": "john@gmail.com",
    "skills": "Python, SQL",
    "education": [],
    "experience": []
}

bad_resume = Resume.model_validate(bad_data)

ValidationError: 1 validation error for Resume
skills
  Input should be a valid list [type=list_type, input_value='Python, SQL', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type

## Conclusion

In this project, I used Gemini to extract information from unstructured resume text and return it as structured JSON.

I then used Pydantic to define the expected structure and validate Gemini's output.

I also tested invalid data and saw that Pydantic raises a ValidationError when the data does not match the schema.

### Key Learning

- Structured output makes LLM responses predictable and easier for programs to process.
- JSON is a structured data format.
- Pydantic allows us to define data models and validate incoming data.
- Pydantic does not train the LLM or change its underlying knowledge.

In [27]:
%%writefile README.md
# Day 04 - Structured Output + Pydantic

## What I learned
- Structured output makes LLM responses predictable and easier for programs to process.
- JSON is a structured data format.
- Pydantic allows us to define expected data structures and validate incoming data.
- Pydantic does not train the LLM or change its underlying knowledge.

## What I built
An AI Resume Extractor using Gemini and Pydantic.

The project takes unstructured resume text and asks Gemini to extract:
- Name
- Email
- Skills
- Education
- Experience

Gemini returns the information as structured JSON, which is then validated using a Pydantic model.

## Validation Test
I tested both valid and invalid data.

Valid data matched the Pydantic schema successfully.

For invalid data, such as providing a string where `list[str]` was expected, Pydantic raised a `ValidationError`.

## Conclusion
This project demonstrated how structured LLM output and Pydantic validation can make AI-generated data more predictable and reliable for Python applications.

Writing README.md
